# D1.11 · Honeypots, canaries and deception in the agent's environment

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.10 · Fleet-level correlation: seeing a swarm](https://spbreed.github.io/cyber-commons/lessons/D1.10.html)**.

| | |
|---|---|
| Tools used | Canarytokens, Inspect, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Authenticate with a canary and watch a zero-threshold alert fire; then salt a benchmark and read the cheat-attempt rate as a leading indicator.

**Why a security engineer needs it.** Every other detector needs a threshold, and every threshold is a trade. Deception needs neither — but only if the bait is placed where the agent actually looks, and rotated before it is learned. The control it builds is: canary tokens in config, environment and artifact metadata (C4.4), and honeypot tasks salted into the benchmark whose cheat path is logged rather than rewarded (C10.3).

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Every detector in this chapter needs a threshold, and every threshold is a trade. A canary needs neither: nothing legitimate has any reason to touch it, so its false-positive rate is zero by construction rather than by tuning.

> **At CyberTravels.** A canary credential in CyberTravels' environment and a honeypot task in the benchmark: two detectors with no threshold to tune, because nothing legitimate has any reason to touch either.

## 2 · The framework

```
   tuned detector                     deception

   threshold ---> TP and FP           canary ---> any touch is a hit
   needs triage                       needs none: nothing legitimate
   finds more                          has a reason to touch it
                                      finds less, at precision 1.00

   canary token in env/config/docs    honeypot task with an instrumented
   real prefix, real length            shortcut that logs and scores 0

   both decay: bait that is not rotated is bait that has been learned
```

Every detector in this chapter has needed a threshold, and every threshold is a
trade between missing things and crying wolf. Deception is the exception:
**a canary has a false-positive rate of zero by construction**, because nothing
legitimate ever touches it.

Two controls from the incident register live here, and both are about the
environment your harness builds rather than the code it analyses.

**Credential canaries (C4.4).** Tokens that look genuine — same prefix, same
format, same length — placed where an agent or an attacker would plausibly
look: config files, environment listings, documentation, artifact metadata. The
issuing service alerts on any authentication attempt. There is no legitimate
use, so a hit is a confirmed compromise and opens an incident automatically. In
the source incident, live credentials found on the public internet were
redistributed to the swarm within minutes; a canary alongside them would have
carried source IP, user agent and timestamp.

**Honeypot tasks (C10.3).** Benchmark tasks salted with a plausible-looking
shortcut that is instrumented rather than functional, plus tasks whose scoring
*appears* manipulable but where manipulation is detected rather than rewarded.
The cheat-attempt rate on those tasks is a leading indicator for the whole
fleet — available continuously, not after the fact.

Both share one operational requirement: **rotate them**. Bait that stays put is
bait that gets learned, and a learned canary is worse than none because it
tells you the environment is clean when it is only well-mapped.

## 3 · The procedure, as a skill

A canary credential nothing legitimate uses produces an alert with no structurally possible false positive. The skill places them, attaches source and user agent to each hit, adds honeypot tasks that measure inclination, and models the decay that sets the rotation interval.

### The skill — [`skills/detection/canary-and-honeypot-design/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/canary-and-honeypot-design/SKILL.md)

```yaml
name: canary-and-honeypot-design
description: >-
  Place credential canaries and honeypot tasks in an agent's environment so that
  an alert has no structurally possible false positive, and measure how fast the
  signal decays as agents learn. Use when deception is being added, or when a
  detection needs to be one nobody has to triage.
allowed-tools: Read, Grep, Glob
```

# An alert with no possible false positive

A canary credential that nothing legitimate uses produces an alert that needs no
triage: the only way it authenticates is that somebody read it and tried it.
That property is structural, not statistical, and it is why deception belongs in
an agent environment where every other signal is ambiguous.

## When to use this

Designing detection for agent environments, and whenever an existing detection's
false-positive rate is the reason it is ignored.

## Procedure

**1 — Place canaries where only reading them is unusual.** Environment
variables, config files, the fixtures a code agent walks. They must be
indistinguishable from real ones — a canary named `canary_key` is a filter, not
a trap.

**2 — Ensure nothing legitimate uses them.** This is the whole property. Check
the code, the tests and the deployment. One legitimate reference and the alert
becomes triage.

**3 — Instrument the authentication path** to capture source address and user
agent on use. A canary alert with no context tells you that it happened and
nothing about who.

**4 — Add honeypot tasks for behaviour rather than credentials.** A task with an
available shortcut that nothing legitimate would take. Log the attempt and score
it; this measures inclination, which no credential can.

**5 — Measure decay.** Agents and operators learn. Model the hit rate over days
since placement, with and without rotation, and set the rotation interval from
the curve rather than from a calendar.

## Output contract

```json
{
  "canaries": [{"id": "str", "placed_in": "str", "indistinguishable": true, "legitimate_refs": 0}],
  "alerts": [{"canary": "str", "source_ip": "str", "user_agent": "str", "false_positive_possible": false}],
  "honeypot_tasks": [{"task": "str", "shortcut": "str", "attempts": 0}],
  "decay": {"days": [0], "hit_rate": [0.0], "rotation_days": 0}
}
```

## Failure modes

- **A canary anything legitimate touches.** The property is gone and the alert
  becomes noise.
- **Naming it as a canary.** It becomes a filter for the competent attacker.
- **Never rotating.** The signal decays and the absence of alerts reads as
  safety.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/canary-and-honeypot-design/scripts/canary_and_honeypot_design.py
SCRIPT = "skills/detection/canary-and-honeypot-design/scripts/canary_and_honeypot_design.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Two canary authentications out of four events are confirmed compromises with source IP and user agent attached, and no false positive is structurally possible. Both honeypot tasks log a cheat attempt and score zero for it. An unrotated canary's detection rate falls to 0% once learned — reporting a clean environment that is only well-mapped — while rotation holds it at 100%. Deception finds fewer things than the volume detectors and finds them at precision 1.00.

## Your turn

Place one canary credential in the environment your agents run in, wired to a real alert, and leave it. The interesting outcome is not the alert; it is discovering, six weeks later, which systems can even see it.

## Where this leaves you

**What you can do now.** Triage as a loop you supervise, detections written for machine-tempo actors, agent telemetry as a real data source, agent-versus-human attribution, drift monitoring — and the two the incident register adds: detections whose subject is the platform, and analytics that read across runs rather than within them.

**What you still cannot do.** Detection ends at the alert. Every lesson here stops one step before the hard part — a fleet that is acting right now, on delegated credentials, faster than the person reading the alert can type.

**Chapter 9 is that step: scope it, contain it, replay it, and decide in advance who is allowed to stop it. Next → D2.1, agent-assisted reconstruction.**

---

**Next → [D2.1 · Agent-assisted reconstruction](https://spbreed.github.io/cyber-commons/lessons/D2.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*